# 07 — Champion Model: XGBoost vs. Baseline (Stage 8)

The question this notebook answers: does XGBoost beat the Stage 7
logistic regression baseline, on the metrics ADR-002 decided actually matter —
not "is XGBoost fancier." Same train set, same test set, same K,
identical scoring functions (`src/modeling/baseline.py`, shared with
Stage 7 for exactly this reason). If it doesn't win, the baseline stays
champion — a legitimate outcome, not a failure to report around.

In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score

from src.config import RAW_CSV_PATH
from src.features.pipeline import run_stage6_split
from src.modeling.baseline import train_logistic_regression, precision_at_k, recall_at_k
from src.modeling.champion import train_xgboost

df = pd.read_csv(RAW_CSV_PATH)
X_train, X_test, y_train, y_test, artifacts = run_stage6_split(df)

# Precise budget-based K, replacing Stage 7's rough "top 20%" approximation
# with ADR-002's actual numbers: ~500 calls against the real 7,043-customer
# base. Stage 7 used an approximation because no model existed yet to
# justify the precision of doing this; now that a real test set exists,
# there's no reason not to use the real ratio.
BUDGET_FRACTION = 500 / 7043
k = max(1, round(len(X_test) * BUDGET_FRACTION))

print(f"Train: {X_train.shape}, Test: {X_test.shape}, K: {k}")

Train: (5634, 17), Test: (1409, 17), K: 100


## Train both models on identical data

In [2]:
lr_model = train_logistic_regression(X_train, y_train)
xgb_model = train_xgboost(X_train, y_train)

lr_scores = lr_model.predict_proba(X_test)[:, 1]
xgb_scores = xgb_model.predict_proba(X_test)[:, 1]
print("Both trained.")

Both trained.


## Head-to-head, identical metrics, identical K

In [3]:
comparison = pd.DataFrame({
    "Logistic Regression": {
        "PR-AUC": average_precision_score(y_test, lr_scores),
        "ROC-AUC (context only)": roc_auc_score(y_test, lr_scores),
        f"Precision@{k}": precision_at_k(y_test.values, lr_scores, k),
        f"Recall@{k}": recall_at_k(y_test.values, lr_scores, k),
    },
    "XGBoost": {
        "PR-AUC": average_precision_score(y_test, xgb_scores),
        "ROC-AUC (context only)": roc_auc_score(y_test, xgb_scores),
        f"Precision@{k}": precision_at_k(y_test.values, xgb_scores, k),
        f"Recall@{k}": recall_at_k(y_test.values, xgb_scores, k),
    },
})
comparison["XGBoost - LR"] = comparison["XGBoost"] - comparison["Logistic Regression"]
comparison.round(4)

,Logistic Regression,XGBoost,XGBoost - LR
PR-AUC,0.6331,0.6466,0.0135
ROC-AUC (context only),0.8379,0.8358,-0.0022
Precision@100,0.7800,0.8100,0.0300
Recall@100,0.2086,0.2166,0.0080


**Verdict:** No tie-break needed — XGBoost won cleanly on all three
metrics that matter: PR-AUC (0.6466 vs 0.6331), Precision@100 (0.810 vs
0.780), and Recall@100 (0.217 vs 0.209). ROC-AUC came out essentially
tied (0.8358 vs 0.8379, a 0.002 gap — noise, not signal, consistent with
ROC-AUC being the context-only metric here). No PR-AUC-vs-Precision@K
tension to resolve this time; XGBoost is simply the better model at
every cutoff that was measured.

Champion: `xgb`

## Feature importance — does XGBoost agree with the baseline on what matters?

Different mechanism than LR's coefficients (gain-based: how much each
feature improves split quality across all trees, not a signed log-odds
effect) — but if the two models fundamentally disagree on what drives
churn, that's worth knowing before trusting either one's answer to "why
do customers churn."

In [4]:
importance = pd.DataFrame({
    "feature": X_train.columns,
    "xgb_importance": xgb_model.feature_importances_,
}).sort_values("xgb_importance", ascending=False).reset_index(drop=True)

importance

,feature,xgb_importance
0,ContractCommitmentMonths,0.287429
1,InternetService_Fiber optic,0.147944
2,PaymentMethod_Electronic check,0.084732
3,StreamingMovies_No internet service,0.078898
4,tenure,0.046523
5,OnlineSecurity_Yes,0.045175
6,StreamingMovies_Yes,0.038759
7,TechSupport_Yes,0.038575
8,StreamingTV_Yes,0.034895
9,PaperlessBilling,0.031922


## Resolving ADR-006's open question: does ContractCommitmentMonths earn its keep?

`ADR-006` kept `ContractCommitmentMonths` over raw `Contract` on
structural grounds alone — ordinality helps models that can use it — and
explicitly said IV couldn't judge whether that was actually true, only a
real model comparison could. This is that comparison.

In [5]:
rank = int(importance.index[importance["feature"] == "ContractCommitmentMonths"][0]) + 1
score = importance.loc[importance["feature"] == "ContractCommitmentMonths", "xgb_importance"].iloc[0]
print(f"ContractCommitmentMonths: rank {rank} of {len(importance)}, importance {score:.4f}")

ContractCommitmentMonths: rank 1 of 17, importance 0.2874


**Verdict:** Confirmed, and more strongly than the synthetic test run
suggested — rank 1 of 17, importance 0.287, nearly double the second-place
feature (`InternetService_Fiber optic` at 0.148). This is real
corroboration of `ADR-006`'s structural bet: keeping the ordinal
`ContractCommitmentMonths` over one-hot `Contract` wasn't just logically
defensible, it turned out to be the single most useful feature in the
entire model.

## Save whichever model wins

Same reasoning as Stage 7's `baseline_logreg.joblib`: Stage 9
(calibration) needs to load the actual winning model, not retrain it
from scratch and risk a different result from a different random split.

In [6]:
import joblib
from src.config import PROJECT_ROOT

MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True)

champion_name = "xgb"  # confirmed: XGBoost wins PR-AUC, Precision@K, and Recall@K
champion_model = {"lr": lr_model, "xgb": xgb_model}[champion_name] if champion_name in ("lr", "xgb") else None

joblib.dump(champion_model, MODELS_DIR / "champion_model.joblib")
print(f"Saved champion model to {(MODELS_DIR / 'champion_model.joblib').relative_to(PROJECT_ROOT)}")

Saved champion model to models\champion_model.joblib


## Stage 8 summary

| Metric | LR | XGBoost | Winner |
|---|---|---|---|
| PR-AUC | 0.6331 | 0.6466 | XGBoost |
| Precision@100 | 0.780 | 0.810 | XGBoost |
| Recall@100 | 0.209 | 0.217 | XGBoost |

**Champion: `xgb`**

## What this notebook does NOT do
- No hyperparameter search — `train_xgboost`'s defaults are reasonable,
  not tuned. Worth the cost only once XGBoost is confirmed the right
  model family to invest further in.
- No expected-value (P(churn) x CLV) ranking — `ADR-002`'s cost math
  treats CLV as roughly constant, so it coincides with plain probability
  ranking for now. Per-customer CLV weighting is a real future
  refinement, not something this stage needed.
- No calibration check on the probabilities themselves — that's Stage 9,
  and it's a different question (is 0.73 an honest 73%) from the ranking
  question this notebook answers.